# Telemetry Analysis

Query historical telemetry into a pandas DataFrame with `TelemetryQuery`, then subscribe to the live SignalR feed with `TelemetryClient` for real-time updates.

In [ ]:
from iam_sdk import IamClient, IamAdminClient
from iam_sdk.jupyter import TelemetryQuery

BASE_URL = "https://localhost:5161"

async with IamClient(BASE_URL) as auth:
    login = await auth.login("admin@example.com", "Password123!")

admin = IamAdminClient(BASE_URL, login.access_token)
query = TelemetryQuery(admin)

In [ ]:
frame = await query.query(metric_name="temperature", device_id="sensor-001", limit=500)
df = frame.to_dataframe()
df.head()

In [ ]:
hourly = await query.aggregate(metric_name="temperature", device_id="sensor-001", aggregation="avg", interval="1h")
hourly_df = hourly.to_dataframe()
hourly_df.plot(x="bucket", y="value", marker="o", title="Hourly avg temperature")

## Live streaming

`TelemetryClient` bridges the SignalR hub into an `asyncio` queue, so it can be consumed with a normal `async for` loop.

In [ ]:
from iam_sdk import TelemetryClient

live = TelemetryClient(BASE_URL, login.access_token)
await live.connect()
live.subscribe_to_device("sensor-001")

count = 0
async for message in live.stream():
    print(message)
    count += 1
    if count >= 5:
        break

await live.disconnect()